<a href="https://colab.research.google.com/github/SpyC0der77/vscolab/blob/master/vscolab_studio_persistent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import subprocess
import zipfile
from pathlib import Path


def _is_valid_vsix(path: Path) -> bool:
    with path.open("rb") as f:
        return f.read(2) == b"PK"


def _extension_id_from_vsix(vsix_path: Path) -> str:
    with zipfile.ZipFile(vsix_path) as zf:
        data = json.loads(zf.read("extension/package.json"))
    return f"{data['publisher']}.{data['name']}-{data['version']}"


def _ensure_vsix(vsix_path: Path, url: str, label: str) -> None:
    if vsix_path.exists() and _is_valid_vsix(vsix_path):
        return
    if vsix_path.exists():
        print(f"Removing invalid VSIX at {vsix_path}", flush=True)
        vsix_path.unlink()
    print(f"Downloading {label}...", flush=True)
    subprocess.run(["wget", "--show-progress", "-O", str(vsix_path), url], check=True)
    if not _is_valid_vsix(vsix_path):
        raise RuntimeError(
            f"Downloaded file at {vsix_path} is not a valid VSIX (expected zip archive)."
        )


def _ensure_server_settings(server_data_dir: Path) -> None:
    settings_dir = server_data_dir / "User"
    settings_dir.mkdir(parents=True, exist_ok=True)
    settings_path = settings_dir / "settings.json"
    settings = {}
    if settings_path.exists():
        settings = json.loads(settings_path.read_text())
    settings["extensions.verifySignature"] = False
    settings_path.write_text(json.dumps(settings, indent=2) + "\n")


def _install_vsix_manually(vsix_path: Path, server_data_dir: Path, ext_id: str) -> None:
    target = server_data_dir / "extensions" / ext_id
    if target.exists():
        return
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(vsix_path) as zf:
        for member in zf.namelist():
            if not member.startswith("extension/") or member.endswith("/"):
                continue
            rel = member[len("extension/") :]
            dest = target / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(zf.read(member))
    print(f"Extension extracted to {target}", flush=True)


def _install_marketplace(server_bin: Path, ext_id: str, server_data_dir: Path) -> None:
    ext_dir = server_data_dir / "extensions"
    if any(ext_dir.glob(f"{ext_id}*")):
        print(f"{ext_id} already installed", flush=True)
        return

    print(f"Installing {ext_id}...", flush=True)
    result = subprocess.run(
        [
            str(server_bin),
            "--install-extension",
            ext_id,
            "--force",
            "--accept-server-license-terms",
            "--server-data-dir",
            str(server_data_dir),
        ],
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="", flush=True)
    if result.stderr:
        print(result.stderr, end="", flush=True)
    if result.returncode != 0:
        raise RuntimeError(f"Failed to install extension {ext_id}.")


def _install_vsix(
    server_bin: Path,
    vsix_path: Path,
    server_data_dir: Path,
    ext_id: str,
) -> None:
    ext_dir = server_data_dir / "extensions" / ext_id
    if ext_dir.exists():
        print(f"{ext_id} already installed at {ext_dir}", flush=True)
        return

    print(f"Installing {ext_id}...", flush=True)
    result = subprocess.run(
        [
            str(server_bin),
            "--install-extension",
            str(vsix_path),
            "--force",
            "--accept-server-license-terms",
            "--server-data-dir",
            str(server_data_dir),
        ],
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="", flush=True)
    if result.stderr:
        print(result.stderr, end="", flush=True)

    if result.returncode == 0:
        return

    print("CLI install failed, extracting VSIX manually...", flush=True)
    _install_vsix_manually(vsix_path, server_data_dir, ext_id)
    if not ext_dir.exists():
        raise RuntimeError(f"Failed to install extension {ext_id}.")


def install_extensions(
    server_bin: Path,
    extensions: list,
    server_data_dir: Path,
    cache_dir: Path,
) -> None:
    if not extensions:
        return

    server_data_dir.mkdir(parents=True, exist_ok=True)
    _ensure_server_settings(server_data_dir)

    for ext in extensions:
        if isinstance(ext, str):
            _install_marketplace(server_bin, ext, server_data_dir)
            continue

        vsix = ext["vsix"]
        vsix_path = Path(vsix)
        if not vsix_path.is_absolute():
            vsix_path = cache_dir / vsix

        label = ext.get("id") or vsix_path.name
        if ext.get("url"):
            if vsix_path.exists() and _is_valid_vsix(vsix_path):
                print(f"Using cached {label} at {vsix_path}", flush=True)
            else:
                _ensure_vsix(vsix_path, ext["url"], label)
        elif not vsix_path.exists():
            raise FileNotFoundError(f"VSIX not found: {vsix_path}")

        ext_id = ext.get("id") or _extension_id_from_vsix(vsix_path)
        _install_vsix(server_bin, vsix_path, server_data_dir, ext_id)

"""Colab Gemini chat bridge for openvscode-server (no GitHub / Copilot auth).

Starts a local OpenAI-compatible proxy over ``google.colab.ai`` and seeds
Continue with a config that points at it. openvscode-server has no GitHub
Copilot Chat; Continue provides the sidebar chat UI instead.
"""

from __future__ import annotations

import json
import threading
import time
import traceback
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from typing import Any, Iterator
from urllib.parse import urlparse

GEMINI_PROXY_PORT = 8787
# Platform-specific VSIX — marketplace ID install often fails on openvscode-server.
CONTINUE_VSIX_VERSION = "1.3.40"
CONTINUE_EXTENSION = {
    "vsix": f"Continue.continue-{CONTINUE_VSIX_VERSION}@linux-x64.vsix",
    "url": (
        "https://open-vsx.org/api/Continue/continue/linux-x64/"
        f"{CONTINUE_VSIX_VERSION}/file/"
        f"Continue.continue-{CONTINUE_VSIX_VERSION}@linux-x64.vsix"
    ),
}
DEFAULT_MODEL = "google/gemini-2.5-flash"


def _messages_to_prompt(messages: list[dict[str, Any]]) -> str:
    parts: list[str] = []
    for msg in messages:
        role = msg.get("role") or "user"
        content = msg.get("content")
        if isinstance(content, list):
            texts = []
            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    texts.append(str(block.get("text") or ""))
                elif isinstance(block, str):
                    texts.append(block)
            content = "\n".join(texts)
        elif content is None:
            content = ""
        else:
            content = str(content)
        if role == "system":
            parts.append(f"System: {content}")
        elif role == "assistant":
            parts.append(f"Assistant: {content}")
        else:
            parts.append(f"User: {content}")
    parts.append("Assistant:")
    return "\n\n".join(parts)


def _list_models() -> list[str]:
    try:
        from google.colab import ai

        models = list(ai.list_models())
        if models:
            return [str(m) for m in models]
    except Exception as exc:
        print(f"colab_chat: list_models failed ({exc}); using default", flush=True)
    return [DEFAULT_MODEL]


def _generate(prompt: str, model: str, stream: bool) -> str | Iterator[str]:
    from google.colab import ai

    kwargs: dict[str, Any] = {"stream": stream}
    if model:
        kwargs["model_name"] = model
    return ai.generate_text(prompt, **kwargs)


def _sse_chunk(model: str, content: str | None, *, finish: str | None = None) -> bytes:
    delta: dict[str, Any] = {}
    if content is not None:
        delta["content"] = content
    choice: dict[str, Any] = {"index": 0, "delta": delta}
    if finish is not None:
        choice["finish_reason"] = finish
    payload = {
        "id": f"chatcmpl-{uuid.uuid4().hex[:24]}",
        "object": "chat.completion.chunk",
        "created": int(time.time()),
        "model": model,
        "choices": [choice],
    }
    return f"data: {json.dumps(payload)}\n\n".encode()


class _GeminiHandler(BaseHTTPRequestHandler):
    server_version = "vscolab-colab-gemini/1.0"

    def log_message(self, fmt: str, *args: Any) -> None:
        print(f"colab_chat proxy: {fmt % args}", flush=True)

    def _send(self, code: int, body: bytes, content_type: str) -> None:
        self.send_response(code)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(body)

    def _json(self, code: int, obj: Any) -> None:
        self._send(code, json.dumps(obj).encode(), "application/json")

    def do_OPTIONS(self) -> None:  # noqa: N802
        self.send_response(204)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "*")
        self.end_headers()

    def do_GET(self) -> None:  # noqa: N802
        path = urlparse(self.path).path.rstrip("/") or "/"
        if path in ("/v1/models", "/models"):
            models = _list_models()
            data = [
                {
                    "id": mid,
                    "object": "model",
                    "created": int(time.time()),
                    "owned_by": "google-colab",
                }
                for mid in models
            ]
            self._json(200, {"object": "list", "data": data})
            return
        if path in ("/health", "/"):
            self._json(200, {"ok": True, "service": "colab-gemini"})
            return
        self._json(404, {"error": {"message": f"Not found: {path}", "type": "invalid_request_error"}})

    def do_POST(self) -> None:  # noqa: N802
        path = urlparse(self.path).path.rstrip("/") or "/"
        if path not in ("/v1/chat/completions", "/chat/completions"):
            self._json(404, {"error": {"message": f"Not found: {path}", "type": "invalid_request_error"}})
            return

        length = int(self.headers.get("Content-Length") or 0)
        raw = self.rfile.read(length) if length else b"{}"
        try:
            body = json.loads(raw.decode() or "{}")
        except json.JSONDecodeError:
            self._json(400, {"error": {"message": "Invalid JSON", "type": "invalid_request_error"}})
            return

        messages = body.get("messages") or []
        model = body.get("model") or DEFAULT_MODEL
        stream = bool(body.get("stream"))
        prompt = _messages_to_prompt(messages)

        try:
            if stream:
                self._stream(model, prompt)
            else:
                text = _generate(prompt, model, stream=False)
                self._json(
                    200,
                    {
                        "id": f"chatcmpl-{uuid.uuid4().hex[:24]}",
                        "object": "chat.completion",
                        "created": int(time.time()),
                        "model": model,
                        "choices": [
                            {
                                "index": 0,
                                "message": {"role": "assistant", "content": str(text)},
                                "finish_reason": "stop",
                            }
                        ],
                        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
                    },
                )
        except Exception as exc:
            traceback.print_exc()
            self._json(
                500,
                {
                    "error": {
                        "message": str(exc),
                        "type": "server_error",
                    }
                },
            )

    def _stream(self, model: str, prompt: str) -> None:
        self.send_response(200)
        self.send_header("Content-Type", "text/event-stream")
        self.send_header("Cache-Control", "no-cache")
        self.send_header("Connection", "keep-alive")
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()

        chunks = _generate(prompt, model, stream=True)
        for piece in chunks:
            if piece is None:
                continue
            self.wfile.write(_sse_chunk(model, str(piece)))
            self.wfile.flush()
        self.wfile.write(_sse_chunk(model, None, finish="stop"))
        self.wfile.write(b"data: [DONE]\n\n")
        self.wfile.flush()


_proxy_lock = threading.Lock()
_proxy_started = False


def start_gemini_proxy(port: int = GEMINI_PROXY_PORT) -> None:
    """Start the OpenAI-compatible Colab Gemini proxy in a daemon thread."""
    global _proxy_started
    with _proxy_lock:
        if _proxy_started:
            print(f"Colab Gemini proxy already running on :{port}", flush=True)
            return

        server = ThreadingHTTPServer(("127.0.0.1", port), _GeminiHandler)

        def _run() -> None:
            print(f"Colab Gemini proxy listening on http://127.0.0.1:{port}/v1", flush=True)
            server.serve_forever()

        thread = threading.Thread(target=_run, name="colab-gemini-proxy", daemon=True)
        thread.start()
        _proxy_started = True


def write_continue_config(port: int = GEMINI_PROXY_PORT, model: str | None = None) -> Path:
    """Write ~/.continue/config.yaml pointing at the local Gemini proxy."""
    models = _list_models()
    chosen = model or (models[0] if models else DEFAULT_MODEL)
    config = f"""name: vscolab Colab Gemini
version: 0.0.1
schema: v1

models:
  - name: Colab Gemini
    provider: openai
    model: {chosen}
    apiBase: http://127.0.0.1:{port}/v1
    apiKey: colab
    roles:
      - chat
      - edit
      - apply
"""
    # Extra models (same proxy) so the picker can switch when Colab exposes several.
    for mid in models:
        if mid == chosen:
            continue
        safe = mid.replace('"', "")
        config += f"""  - name: {safe}
    provider: openai
    model: {safe}
    apiBase: http://127.0.0.1:{port}/v1
    apiKey: colab
    roles:
      - chat
      - edit
"""

    continue_dir = Path.home() / ".continue"
    continue_dir.mkdir(parents=True, exist_ok=True)
    path = continue_dir / "config.yaml"
    path.write_text(config, encoding="utf-8")
    print(f"Wrote Continue config → {path} (model={chosen})", flush=True)
    return path


def setup_colab_chat(port: int = GEMINI_PROXY_PORT) -> None:
    """Start proxy + seed Continue. Call after extensions are installed."""
    try:
        start_gemini_proxy(port)
        write_continue_config(port)
        print(
            "Chat ready: open the Continue sidebar in VS Code "
            "(no GitHub sign-in; uses Colab Gemini).",
            flush=True,
        )
    except Exception as exc:
        traceback.print_exc()
        print(f"colab_chat setup failed: {exc}", flush=True)

"""Google Drive persistence + openvscode-server bootstrap for vscolab.

Syncs the VS Code ``--default-folder`` path with ``MyDrive/vscolab/`` on Drive.
Load: pull once (Drive -> workspace). Runtime: push-only background sync.
Pre-installs extensions from ``EXTENSIONS`` (marketplace IDs and/or VSIX).
"""

import subprocess
import threading
import time
from pathlib import Path

from google.colab import drive, output

SYNC_INTERVAL = 5
DRIVE_STORE = Path("/content/drive/MyDrive/vscolab")
IGNORE_FILE = ".vscolabignore"
DEFAULT_IGNORE = """\
# Patterns here stay on the Colab VM only — never synced to Drive.
__pycache__/
*.py[cod]
node_modules/
.venv/
venv/
*.egg-info/
.git/
"""

VERSION = "openvscode-server-v1.109.5"
PORT = 3000
GIT_REPO = ""
VSCOLAB_RAW = "https://github.com/SpyC0der77/vscolab/raw/master"
EXTENSIONS = [
    CONTINUE_EXTENSION,
    {
        "vsix": "easy-installer-1.0.0.vsix",
        "url": f"{VSCOLAB_RAW}/extensions/easy-installer/easy-installer-1.0.0.vsix",
    },
    # Marketplace IDs:
    # "ms-python.python",
]


class Persistence:
    def __init__(self, workspace: Path):
        self.workspace = workspace.resolve()

    @property
    def drive_store(self) -> Path:
        return DRIVE_STORE

    @property
    def data_dir(self) -> Path:
        return DRIVE_STORE / "data"

    @property
    def cache_dir(self) -> Path:
        return DRIVE_STORE / "cache"

    def mount(self) -> None:
        print("Mounting Google Drive...", flush=True)
        drive.mount("/content/drive")
        for d in (self.drive_store, self.data_dir, self.cache_dir):
            d.mkdir(parents=True, exist_ok=True)
        print(f"vscolab folder: {self.drive_store}", flush=True)

    def _ignore_file(self) -> Path:
        path = self.drive_store / IGNORE_FILE
        if not path.exists():
            path.write_text(DEFAULT_IGNORE)
        return path

    def _sync_ignore_for_pull(self) -> None:
        self.workspace.mkdir(parents=True, exist_ok=True)
        (self.workspace / IGNORE_FILE).write_text(self._ignore_file().read_text())

    def _sync_ignore_for_push(self) -> None:
        ws_ignore = self.workspace / IGNORE_FILE
        drive_ignore = self._ignore_file()
        if ws_ignore.exists():
            drive_ignore.write_text(ws_ignore.read_text())
        else:
            ws_ignore.write_text(drive_ignore.read_text())

    def _ensure_writable(self) -> None:
        subprocess.run(["chmod", "-R", "u+w", str(self.workspace)], check=False)

    def pull(self) -> None:
        print(f"Pulling from Drive into {self.workspace}...", flush=True)
        self._sync_ignore_for_pull()
        subprocess.run(
            [
                "rsync", "-rl",
                "--no-perms", "--no-owner", "--no-group",
                "--filter=:- .vscolabignore",
                "--exclude=data/", "--exclude=cache/",
                f"{self.drive_store}/", f"{self.workspace}/",
            ],
            check=False,
        )
        self._ensure_writable()

    def push(self) -> None:
        self._sync_ignore_for_push()
        subprocess.run(
            [
                "ionice", "-c3", "nice", "-n", "19",
                "rsync", "-rl",
                "--size-only",
                "--no-perms", "--no-owner", "--no-group",
                "--filter=:- .vscolabignore",
                "--delete", "--delete-excluded",
                "--exclude=data/", "--exclude=cache/",
                f"{self.workspace}/", f"{self.drive_store}/",
            ],
            check=False,
        )

    def start_push_loop(self) -> None:
        def loop():
            while True:
                time.sleep(SYNC_INTERVAL)
                self.push()

        threading.Thread(target=loop, daemon=True).start()
        print(
            f"Background push every {SYNC_INTERVAL}s -> Drive (see {IGNORE_FILE})",
            flush=True,
        )


folder = Path("/content/workspace")
if GIT_REPO:
    name = GIT_REPO.rstrip("/").removesuffix(".git").split("/")[-1]
    folder = Path(f"/content/{name}")

p = Persistence(workspace=folder)
p.mount()
p.pull()

if GIT_REPO and not folder.exists():
    print(f"Cloning {GIT_REPO}...", flush=True)
    subprocess.run(["git", "clone", "--progress", GIT_REPO, str(folder)], check=True)
    p.push()

url = f"https://github.com/gitpod-io/openvscode-server/releases/download/{VERSION}/{VERSION}-linux-x64.tar.gz"
tarball = f"{VERSION}-linux-x64.tar.gz"
tarball_path = p.cache_dir / tarball
local_server = Path(f"/content/{VERSION}-linux-x64")
server_bin = local_server / "bin/openvscode-server"

if not tarball_path.exists():
    print("Downloading openvscode-server...", flush=True)
    subprocess.run(["wget", "--show-progress", "-O", str(tarball_path), url], check=True)
else:
    print(f"Using cached tarball at {tarball_path}", flush=True)

if not local_server.exists():
    print("Extracting...", flush=True)
    subprocess.run(["tar", "-xzf", str(tarball_path), "-C", "/content"], check=True)
else:
    print(f"Using extracted server at {local_server}", flush=True)

install_extensions(server_bin, EXTENSIONS, p.data_dir, p.cache_dir)
setup_colab_chat()

p.push()
p.start_push_loop()

folder = str(folder.resolve())

print(f"Starting openvscode-server (default folder: {folder})...", flush=True)
subprocess.Popen([
    str(server_bin),
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--without-connection-token",
    "--accept-server-license-terms",
    "--server-data-dir", str(p.data_dir),
    "--default-folder", folder,
])
time.sleep(5)
print(f"openvscode-server running on port {PORT} — {folder}", flush=True)
print(f"Drive storage: {p.drive_store}", flush=True)

# CELL 2
url = output.eval_js(f'google.colab.kernel.proxyPort({PORT}, {{"cache": false}})')
print(f"Open VS Code: {url}", flush=True)